In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}

div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

## 2.2 1500글자씩 겹쳐 쪼개 읽어오기

In [5]:
import time
start= time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # character 단위로 짜름
loader = Docx2txtLoader('tax_docx/소득세법(법률)(제21221호)(20260701).docx')
text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500, #문서를 쪼갤 떄 1500글자씩 chunking
        chunk_overlap=20, #200문자씩 overlap 하여 chunking 작업
        #separators=["\n\n","\n"," "] -> 기본값   : 다음의 순서대로 chunking 시도
)
documents = loader.load_and_split(text_splitter=text_splitter)
runtime=time.time()-start
print('문서를 1500글자로 겹쳐 쪼개면서 읽는 시간: ', runtime)
print('chunk 갯수: ', len(documents))

문서를 1500글자로 겹쳐 쪼개면서 읽는 시간:  4.959320545196533
chunk 갯수:  180


# 3. 쪼갠 문서를 임베딩->벡터 데이터 베이스(파일)에 저장
- 임베딩 모델 :upstage의 embedding-query, solar-embedding-1-large-passage
- vector DB 저장 폴더 : ./chroma

In [6]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding= UpstageEmbeddings(
            #api_key=os.getenv('OPENAI_API_KEY'),
            model='solar-embedding-1-large-passage'    
            )


In [ ]:
# embed_query(): 한 문자열을 임베딩 벡터로 전환
embedding.embed_query("소득세법은 어쩌구 저쩌구")

In [ ]:
len(embedding.embed_query("소득세법은 어쩌구 저쩌구"))

In [ ]:
#embed_documents(): 여러 문자열을 임베딩 벡터로 전환
vectors=embedding.embed_documents([(document.page_content) for document in documents])

In [ ]:
print(len(vectors[0]))

In [9]:
%%time
from langchain_chroma import Chroma
# 데이터를 처음 저장할 때 
# database= Chroma.from_documents(
#         documents=documents, #chunking 작업한 documents
#         embedding=embedding, #임베딩 객체
#         collection_name='tax-collection', # 생략시 이름이 랜덤
#         persist_directory='./chroma_upstage'      # 생략시 로컬에 저장 안됨. 인메모리 상태
# )
# 저장된 vector DB를 가져올 때 
database= Chroma(
    embedding_function=embedding,
    collection_name='tax-collection',
    persist_directory='./chroma_upstage'
)

CPU times: total: 31.2 ms
Wall time: 25.6 ms


In [10]:
# 확인
results =database.get(include=['embeddings','documents','metadatas'])

In [11]:
print('데이터수 : ', len(results['ids']))
print('문서 임베딩 차원 수 :', len(results['embeddings'][0]))
print('0번째 원본:',results['documents'][1][:100])
print('1번째 원본의 임베딩 벡터:',results['embeddings'][1][:10])
print('1번째 metadata:', results['metadatas'][1])


데이터수 :  180
문서 임베딩 차원 수 : 4096
0번째 원본: 1. 구성원 간 이익의 분배비율이 정하여져 있고 해당 구성원별로 이익의 분배비율이 확인되는 경우

2. 구성원 간 이익의 분배비율이 정하여져 있지 아니하나 사실상 구성원별로 이익이
1번째 원본의 임베딩 벡터: [-0.01432037  0.00081682 -0.01189423 -0.00717163 -0.01140594  0.0029335
 -0.01353455 -0.01026917 -0.00888062 -0.00572586]
1번째 metadata: {'source': 'tax_docx/소득세법(법률)(제21221호)(20260701).docx'}


# 4. vector DB에 질문과 유사도 검색(retrieval)

In [12]:
query = "연봉 5천만원인 직장인의 소득세는 얼마예요?"
retrieved_docs = database.similarity_search(query=query,
                                           k=2) # 유사도 2개  // 기본 k값은 4


In [15]:
# 검색된 문서를 LLM에 전달하기 위한 가공
retrieved_doc='\n--\n'.join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])
retrieved_doc[:100]

'② 제1항의 경우에 제56조, 제56조의2, 제57조 및 제57조의2에 따라 공제할 세액이 있을 때에는 이를 공제한 후의 세액을 소득세액으로 하여 제1항을 적용한다.<개정 2022'

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변 생성-1
langchain 기법X

In [16]:
from langchain_openai import ChatOpenAI
load_dotenv()
llm=ChatOpenAI(model='gpt-4.1-nano')


In [17]:
from langchain_upstage import ChatUpstage
load_dotenv()
llm=ChatUpstage(model='solar-pro4')

In [18]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다
- [context]를 참조해서 사용자의 질문에 답변해 주세요. context는 다음과 같습니다
{retrieved_doc}
질문 : {query}"""
ai_message=llm.invoke(prompt)

In [19]:
ai_message.usage_metadata

{'input_tokens': 1636,
 'output_tokens': 971,
 'total_tokens': 2607,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [20]:
print(ai_message.content)

안녕하십니까? 저는 한국의 소득세법 전문가로서 귀하의 질문에 답변해 드리겠습니다.

결론부터 말씀드리면, **제공된 법령 정보만으로는 연봉 5,000만원인 직장인의 소득세를 정확히 산출할 수 없습니다.**

소득세를 계산하기 위해서는 단순히 연봉(총급여액)뿐만 아니라 **근로소득공제, 기본공제, 세액공제 등** 여러 단계의 공제액을 순차적으로 적용해야 하기 때문입니다. 또한, 연봉 외에 다른 소득(추가 근무, 이자, 배당 등)이 있는지, 부양가족이 있는지, 지출한 특별공제 항목(신용카드 등)이 있는지에 따라 최종 세액이 달라집니다.

다만, 귀하께서 제공해 주신 소득세법 조문들을 바탕으로 **연봉 5,000만원 직장인의 세액 계산 구조와 적용 법리**를 다음과 같이 설명해 드릴 수 있습니다.

### 1. 근로소득세 계산 기본 구조 (산식)

소득세는 일반적으로 다음 순서로 계산됩니다.
1.  **총급여액:** 연봉 (5,000만원 가정)
2.  **근로소득공제:** 총급여액에서 일정 금액을 공제 (제공된 조문 중 근로소득공제 관련 부분 참조)
3.  **종합소득과세표준:** (총급여액 - 근로소득공제) - 종합소득공제(기본공제, 추가공제 등)
4.  **종합소득산출세액:** 과세표준 × 기본세율(6%~45%)
5.  **결정세액:** 산출세액 - 세액공제(근로소득세액공제 등)

### 2. 제공 법령 기반 적용 사항 검토

**① 근로소득공제 (제47조 등 참조 필요)**
- 제공된 맥락에는 근로소득공제의 구체적 계산표(제47조 제1항)가 생략되어 있으나, 일반적으로 연봉 5,000만원 구간에서는 총급여액이 4,500만원 초과 1억원 이하 구간에 해당하여 일정 금액(예: 1,200만원 + 4,500만원 초과분의 5%)을 공제하게 됩니다. 이로 인해 과세표준이 되는 근로소득금액이 줄어듭니다.

**② 종합소득공제 - 기본공제 (제50조)**
- 제50조 제1항에 따라 **본인**의 기본공제(연 150만원)는 반드시 적용됩니다. (다만, 배우자의 소득 유무, 부

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변 생성-2

In [21]:
from langchain_core.prompts import ChatPromptTemplate
promptTemplate = ChatPromptTemplate([
                     ('system', '당신은 최고의 한국 소득세법 전문가입니다'),
                     ('human', f'''다음 문맥을 참고하여 질문에 답변하세요. 답을 모르면 모른다고 답변하세요.
    최대 3문장으로 간결하게 답변하세요.
    질문 : {{question}}
    문맥 : {{context}}
    답변 : ''')
                    ]) 

In [22]:
prompt = promptTemplate.invoke({'context':retrieved_doc, 
                                'question':query})

In [23]:
#과금
ai_message = llm.invoke(prompt)

In [24]:
from langchain_core.output_parsers import StrOutputParser
outputParser = StrOutputParser()
outputParser.invoke(ai_message)

'제공된 문맥만으로는 연봉 5천만원 직장인의 정확한 소득세 금액을 계산할 수 없으므로 답변드리기 어렵습니다. 소득세액은 총급여액 외에도 근로소득공제, 기본공제, 인적공제, 세액공제 등 다양한 공제 항목과 적용 대상에 따라 달라지기 때문입니다. 정확한 세액 계산을 위해서는 관할 세무서나 전문가를 통해 종합적인 소득 및 공제 사항을 확인하시기 바랍니다.'

In [25]:
# 위의 작업을 한번에
outputParser.invoke(llm.invoke(promptTemplate.invoke(
    {'context':retrieved_doc,'question':query}
)))

'제공된 문맥에는 근로소득세 산정의 기본 기준이 되는 종합소득세율표와 과세표준 계산 방식이 포함되어 있지 않아 정확한 세액을 산출할 수 없습니다. 참고로 연봉 5천만원(총급여 기준)의 경우 근로소득세액공제는 약 6만원, 기본공제(본인 1인)는 150만원이 적용되지만, 이는 최종 소득세액이 아닌 중간 공제 항목에 불과합니다. 실제 소득세는 과세표준(총급여에서 각종 공제를 차감한 금액)에 기본세율(6~45%)을 적용한 산출세액에서 세액공제 항목을 공제하여 결정되므로, 정확한 금액은 국세청 홈택스의 세금 모의계산 서비스를 이용하셔야 합니다.'

# 6. langchain으로 답변 생성

In [26]:
rag_chain =promptTemplate | llm | outputParser
rag_chain.invoke({'context':retrieved_doc, 'question': query})


'제시된 문맥만으로는 제공할 수 없습니다. 문맥에는 근로소득세액공제 계산식(총급여 5천만 원 시 약 59만 6천 원)과 종합소득세율표가 누락되어 있어 연간 소득세를 정확히 산출할 수 없습니다. 정확한 금액은 부양가족 수, 4대 보험료 및 신용카드 공제 등 종합소득공제·세액공제를 반영한 연말정산 결과로 확인해야 합니다.'

Langchain으로 전달하기

In [27]:
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [28]:
# 1. 임베딩 & LLM 초기화
load_dotenv()
embedding = UpstageEmbeddings(model = "solar-embedding-1-large-passage")
llm = ChatUpstage(model = 'solar-pro4')
#2. vector DB load
vectorStore = Chroma(
    embedding_function=embedding,
    persist_directory='./chroma_upstage',
    collection_name='tax-collection'
)
#3. Retriever 생성 
# (1) search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(
    search_type='similarity',
    search_kwargs={'k':3} # 유사도가 높은 top-3 추출
)
# # (2) search_type='mmr' : 후보 10개중 서로 덜 겹치는 3개를 추출
# retriever = vectorStore.as_retriever(
#     search_type='mmr',
#     search_kwargs={'k':3,
#                   'fetch_k':10,
#                   'lambda_mult':0.5} # 1에 가까울수록 관련성 깊은 계산 
# )

# # (3) search_type='similarity_score_threshold' : 
# retriever = vectorStore.as_retriever(
#     search_type='similarity_score_threshold',
#     search_kwargs={'k':3, 'score_threshold':0.4} 
# )

#4. 프롬프트템플릿
template= f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질몬에 답하세요.
답을 모르면 모른다고 답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 : """
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return '\n--\n'.join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])

In [29]:
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever|format_documents,
        "query" : RunnablePassthrough() # 질문만 그대로 전달
    }
    |prompt
    |llm
    |StrOutputParser()
)
rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')

'제공된 문맥은 재해손실세액공제, 근로소득세액공제, 자녀세액공제 등 일부 공제 조항만 담고 있을 뿐, 소득세율표, 다른 소득공제 항목, 실제 공제 및 감면 적용 내역이 누락되어 있어 정확히 계산할 수 없습니다. 연소득 5천만 원의 직장인이 납부할 실제 소득세는 총급여액에서 근로소득공제·기본공제 등을 뺀 과세표준에 기본세율을 적용한 뒤, 각종 세액공제를 차감해야 도출됩니다. 따라서 가족 구성, 4대 보험, 신용카드 사용액 등 추가 정보가 필요하며, 주어진 내용만으로는 금액을 특정할 수 없습니다.'